In [ ]:
import os, random, pickle, warnings, time, glob
from pathlib import Path
import numpy as np, pandas as pd, rasterio
from rasterio.warp import transform
from scipy import ndimage
from sklearn.metrics import f1_score, precision_score, recall_score
import lightgbm as lgb

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
PROJECT = Path("../..")
ROOT = PROJECT / "af"
TEST = PROJECT / "fire-test-renamed" / "test"
FIRMS_DIR = PROJECT / "ADDITIONAL_DATA"
MASKS = ROOT / "masks"
WINS = (5, 11, 21)
NEG_PER_CHIP = 1500
FLARE_SHARE = 0.30
FLARE_WEIGHT = 10.0
N_FOLDS = 5
GSD, SIZE = 375.0, 256
THRESHOLDS = [0.9, 0.99, 0.995, 0.999, 0.9995, 0.9999]
PARAMS = dict(
    n_estimators=500,
    learning_rate=0.08,
    num_leaves=63,
    min_child_samples=40,
    reg_lambda=1.0,
    random_state=SEED,
    verbose=-1,
    n_jobs=-1,
)
CHANNELS = [
    "I1",
    "I2",
    "I3",
    "I4",
    "I5",
    "solar_zenith",
    "sensor_zenith",
    "valid",
    "landcover",
    "dem",
    "t2m",
    "rh2m",
    "wind_speed",
]
meta = pd.read_csv(ROOT / "meta.csv")
meta["tile"] = list(zip(meta.epsg, meta.x_min, meta.y_min))
IDS = meta.chip_id.tolist()
print("чипов:", len(IDS), "| seed:", SEED)

чипов: 420 | seed: 42


In [ ]:
def window_stats(a, w):
    mean = ndimage.uniform_filter(a, w, mode="reflect")
    sq = ndimage.uniform_filter(a * a, w, mode="reflect")
    return mean, np.sqrt(np.maximum(sq - mean * mean, 1e-6))


def pixel_features(channels):
    v = {k: np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0) for k, a in channels.items()}
    I4, I5 = v["I4"], v["I5"]
    diff = I4 - I5
    cols = [v[k] for k in CHANNELS]
    names = list(CHANNELS)
    cols += [diff, I4 / np.maximum(I5, 1e-3), v["I3"] - v["I2"]]
    names += ["diff", "I4_over_I5", "I3_minus_I2"]
    for w in WINS:
        for band, tag in ((I4, "I4"), (diff, "diff")):
            mean, std = window_stats(band, w)
            cols += [band - mean, (band - mean) / std, std]
            names += [f"{tag}_dev{w}", f"{tag}_z{w}", f"{tag}_std{w}"]
    cols += [
        ndimage.maximum_filter(diff, 3) - diff,
        ndimage.maximum_filter(I4, 3) - I4,
        ndimage.minimum_filter(I4, 3),
        ndimage.median_filter(I4, 5),
    ]
    names += ["diff_peak_gap", "I4_peak_gap", "I4_min3", "I4_median5"]
    return np.stack(cols).reshape(len(cols), -1).T.astype(np.float32), names


def read_channels(chip_id, root=ROOT):
    out = {}
    for sub, fname in (("viirs", f"{chip_id}_VIIRS_I1-I5.tif"), ("aux", f"{chip_id}_AUX.tif")):
        with rasterio.open(root / sub / fname) as src:
            for i, name in enumerate(src.descriptions, 1):
                out[name] = src.read(i).astype(np.float32)
    return out


def read_mask(chip_id):
    with rasterio.open(MASKS / f"{chip_id}_MASK.tif") as src:
        return (src.read(1) == 1).ravel()


FEATURES = {c: pixel_features(read_channels(c))[0] for c in IDS}
NAMES = pixel_features(read_channels(IDS[0]))[1]
TARGETS = {c: read_mask(c) for c in IDS}
print(f"признаков: {len(NAMES)} | горящих пикселей: {sum(int(t.sum()) for t in TARGETS.values())}")

признаков: 38 | горящих пикселей: 9725


In [ ]:
firms = pd.concat(
    [pd.read_csv(f, low_memory=False) for f in sorted(FIRMS_DIR.glob("*/fire_archive_*.csv"))]
)
firms = firms.drop_duplicates(subset=["latitude", "longitude", "acq_date", "acq_time", "satellite"])
firms["acq_date"] = pd.to_datetime(firms.acq_date)
winter = firms.acq_date.dt.month.isin([12, 1, 2]).to_numpy()
year = firms.acq_date.dt.year.to_numpy()
projected = {
    ep: np.array(
        transform("EPSG:4326", f"EPSG:{ep}", firms.longitude.tolist(), firms.latitude.tolist())
    )
    for ep in (32637, 32638)
}

FLARES = {}
for r in meta.itertuples():
    x, y = projected[int(r.epsg)]
    col = ((x - r.x_min) / GSD).astype(np.int32)
    row = ((r.y_max - y) / GSD).astype(np.int32)
    inside = (col >= 0) & (col < SIZE) & (row >= 0) & (row < SIZE)
    idx = row[inside] * SIZE + col[inside]
    winter_hits = np.bincount(idx[winter[inside]], minlength=SIZE * SIZE)
    years_seen = sum(
        (np.bincount(idx[year[inside] == yv], minlength=SIZE * SIZE) > 0).astype(int)
        for yv in range(2019, 2026)
    )
    FLARES[r.chip_id] = (winter_hits >= 1) & (years_seen >= 3)

flare_px = sum(int(f.sum()) for f in FLARES.values())
print(f"детекций FIRMS: {len(firms)} | зимних: {int(winter.sum())}")
print(f"пикселей-кандидатов в факелы: {flare_px}")

детекций FIRMS: 1218415 | зимних: 36439
пикселей-кандидатов в факелы: 24361


In [ ]:
tile_key = np.array([str(t) for t in meta.tile])
order = sorted(set(tile_key))
np.random.RandomState(SEED).shuffle(order)
FOLDS = [set(order[i::N_FOLDS]) for i in range(N_FOLDS)]


def training_set(chip_ids, seed=SEED):
    rng = np.random.RandomState(seed)
    X, y, w = [], [], []
    for c in chip_ids:
        target, flare = TARGETS[c], FLARES[c]
        pos = np.flatnonzero(target)
        neg_flare = np.flatnonzero(~target & flare)
        neg_other = np.flatnonzero(~target & ~flare)
        take_flare = rng.choice(
            neg_flare, min(int(NEG_PER_CHIP * FLARE_SHARE), neg_flare.size), replace=False
        )
        take_other = rng.choice(
            neg_other, min(NEG_PER_CHIP - take_flare.size, neg_other.size), replace=False
        )
        idx = np.concatenate([pos, take_flare, take_other])
        weight = np.ones(idx.size, np.float32)
        weight[pos.size : pos.size + take_flare.size] = FLARE_WEIGHT
        X.append(FEATURES[c][idx])
        y.append(target[idx])
        w.append(weight)
    return np.concatenate(X), np.concatenate(y), np.concatenate(w)


oof = np.zeros(len(IDS) * SIZE * SIZE, np.float32)
truth = np.concatenate([TARGETS[c] for c in IDS])
position = {c: i for i, c in enumerate(IDS)}
for fold, val_tiles in enumerate(FOLDS, 1):
    train_ids = [c for c, t in zip(IDS, tile_key) if t not in val_tiles]
    val_ids = [c for c, t in zip(IDS, tile_key) if t in val_tiles]
    X, y, w = training_set(train_ids)
    model = lgb.LGBMClassifier(**PARAMS).fit(X, y, sample_weight=w)
    for c in val_ids:
        s = position[c] * SIZE * SIZE
        oof[s : s + SIZE * SIZE] = model.predict_proba(FEATURES[c])[:, 1]
    print(f"  фолд {fold}/{N_FOLDS}: train {len(train_ids)}, val {len(val_ids)}", flush=True)

scores = pd.DataFrame(
    [
        (
            t,
            precision_score(truth, oof >= t),
            recall_score(truth, oof >= t),
            f1_score(truth, oof >= t),
        )
        for t in THRESHOLDS
    ],
    columns=["threshold", "precision", "recall", "f1"],
)
THRESHOLD = float(scores.loc[scores.f1.idxmax(), "threshold"])
print(scores.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nпорог {THRESHOLD} -> F1 = {scores.f1.max():.4f}")

  фолд 1/5: train 346, val 74
  фолд 2/5: train 312, val 108
  фолд 3/5: train 369, val 51
  фолд 4/5: train 301, val 119
  фолд 5/5: train 352, val 68
 threshold  precision  recall     f1
    0.9000     0.8636  0.9900 0.9225
    0.9900     0.8926  0.9758 0.9324
    0.9950     0.9032  0.9681 0.9345
    0.9990     0.9216  0.9342 0.9278
    0.9995     0.9313  0.9147 0.9229
    0.9999     0.9498  0.8461 0.8949

порог 0.995 -> F1 = 0.9345


In [ ]:
def rle_encode(mask):
    flat = np.concatenate([[0], mask.ravel(order="C").astype(np.uint8), [0]])
    edges = np.flatnonzero(flat[1:] != flat[:-1]) + 1
    return " ".join(f"{s} {e - s}" for s, e in zip(edges[0::2], edges[1::2]))


def rle_decode(text, shape=(SIZE, SIZE)):
    out = np.zeros(shape[0] * shape[1], bool)
    if isinstance(text, str) and text.strip():
        v = list(map(int, text.split()))
        for start, length in zip(v[0::2], v[1::2]):
            out[start - 1 : start - 1 + length] = True
    return out.reshape(shape, order="C")


assert all(
    np.array_equal(
        rle_decode(rle_encode(TARGETS[c].reshape(SIZE, SIZE))), TARGETS[c].reshape(SIZE, SIZE)
    )
    for c in IDS
)

X, y, w = training_set(IDS)
model = lgb.LGBMClassifier(**PARAMS).fit(X, y, sample_weight=w)
pickle.dump(
    {"model": model, "threshold": THRESHOLD, "features": NAMES, "windows": WINS, "seed": SEED},
    open("model.pkl", "wb"),
)

template = pd.read_csv(TEST / "sample_submission.csv")
af_chips = template.loc[template.chip_id.str.startswith("AF_"), "chip_id"].unique().tolist()
records = []
for chip_id in af_chips:
    X_test, _ = pixel_features(read_channels(chip_id, root=TEST / "af"))
    mask = (model.predict_proba(X_test)[:, 1] >= THRESHOLD).reshape(SIZE, SIZE)
    records.append({"chip_id": chip_id, "class_id": 1, "rle": rle_encode(mask)})
submission = pd.DataFrame(records, columns=["chip_id", "class_id", "rle"])
submission["rle"] = submission["rle"].fillna("").astype(str)
assert submission.notna().all().all()
assert set(submission.chip_id) == set(af_chips)
submission.to_csv("submission.csv", index=False)
total = sum(sum(map(int, r.split()[1::2])) for r in submission.rle if r)
print(
    f"submission.csv: {len(submission)} строк, непустых {int((submission.rle.str.len() > 0).sum())}"
)
print(f"пикселей горения: {total} ({total / len(submission):.0f} на чип)")

submission.csv: 180 строк, непустых 131
пикселей горения: 3877 (22 на чип)
